In [1]:
import numpy as np
import pandas as pd
from gprofiler import GProfiler
import gseapy as gsp
import os

In [7]:
# Config 
DATASET = "Zebrafish-2024"  # Change this to "Zebrafish-2024" or "Drosophila-2025" as needed
DATASET_CONFIG = {
    "Mouse-2023": {
        "model_org": "mmusculus",
        "comparisons": ["CFA_vs_control", "CFB_vs_control"]
    },
    "Zebrafish-2024": {
        "model_org": "drerio",
        "comparisons": ["CNR401_vs_BMAA", "CFA_vs_BMAA", "Edaravone_vs_BMAA"]
    },
    "Drosophila-2025": {
        "model_org": "dmelanogaster",
        "comparisons": ["CNR401_vs_control", "CFA_vs_control"]
    }
}

DEA_DIR = os.path.join(DATASET, "DEA-results")
OUT_GSEA = os.path.join(DATASET, "raw-GSEA-results")
OUT_DIR = os.path.join(DATASET, "GSEA-results")

ALPHA = 0.05
TOP_N = 200   

# g:Profiler settings
gp = GProfiler(return_dataframe=True)
MODEL_ORG = DATASET_CONFIG[DATASET]["model_org"]
TARGET_ORG = "hsapiens"
Comparisons = DATASET_CONFIG[DATASET]["comparisons"]

# gseapy settings
# https://maayanlab.cloud/Enrichr/#libraries
GSEA_LIBS = [
    'GO_Biological_Process_2025', 
    'GO_Molecular_Function_2025', 
    'GO_Cellular_Component_2025',
    'KEGG_2021_Human', 
    'Reactome_2022', 
    'WikiPathways_2024_Human'
]
PERMUTATIONS = 1000

In [ ]:
gene_lists = {}
for comparison in Comparisons:
  # read DEG file 
  file_path = os.path.join(DEA_DIR, f"DEG_{comparison}.csv")
  df = pd.read_csv(file_path, index_col="gene_id")
  # map to human orthologs 
  all_genes = df.index.tolist()
  orths = gp.orth(organism=MODEL_ORG, query=all_genes, target=TARGET_ORG)
  mapping_df = orths[['incoming', 'ortholog_ensg', 'name', 'description']].copy()
  # collapse model org genes that map to different orthologs (one-to-many)
  model_collapsed = mapping_df.groupby('incoming').agg({
    'ortholog_ensg': lambda x: '; '.join(set(x.dropna())),
    'name': lambda x: '; '.join(set(x.dropna())),
    'description': lambda x: '; '.join(set(x.dropna()))
  }).reset_index()  
  # Merge with gene-level DEA results
  model_df = df.merge(model_collapsed, left_index=True, right_on='incoming', how='left')
  model_df = model_df.set_index('incoming')
  model_df.index.name = 'gene_id'
  out_path = os.path.join(DEA_DIR, f"model_orthologs_{comparison}.csv")
  model_df.to_csv(out_path)
  # collapse human genes that map from different model org genes (many-to-one)
  human_mapping = mapping_df[mapping_df['ortholog_ensg'] != "N/A"]
  human_df = human_mapping.merge(df, left_on='incoming', right_index=True, how='inner')
  human_df['abs_stat'] = human_df['stat'].abs()
  human_df = human_df.sort_values(['ortholog_ensg', 'abs_stat'], ascending=[True, False])
  human_df_collapsed = human_df.groupby('ortholog_ensg').agg({
    'name': 'first',
    'description': 'first',
    'stat': 'first',
    'baseMean': 'first',
    'LFC_raw': 'first',
    'LFC_shrunk': 'first',
    'pvalue': 'first',
    'padj': 'first',
    'incoming': lambda x: '; '.join(x) # All model genes, most extreme first
    })
  human_df_collapsed = human_df_collapsed.rename(columns={'incoming': DATASET_CONFIG[DATASET]["model_org"] + '_genes'})
  human_df_collapsed = human_df_collapsed.reset_index().set_index('name')
  out_path = os.path.join(DEA_DIR, f"human_orthologs_{comparison}.csv")
  human_df_collapsed.to_csv(out_path)
  # extract ranked list of human gene names for GSEA``
  gene_list = human_df_collapsed[human_df_collapsed.index.notnull()] # drop unannotated genes
  gene_list = gene_list['stat'].dropna().sort_values(ascending=False)
  out_path = os.path.join(OUT_DIR, f"gene_list_{comparison}.csv")
  gene_list.to_csv(out_path)

In [ ]:
# Some genes have duplicate stat values 
# This is the result of mapping from the same gene
# Introduce a small amount of noise to break ties 
gene_lists = {}
np.random.seed(42)
for comparison in Comparisons:
  file_path = os.path.join(OUT_DIR, f"gene_list_{comparison}.csv")
  gene_list = pd.read_csv(file_path, index_col='name').squeeze("columns")
  duplicate_mask = gene_list.duplicated(keep=False)
  noise = np.random.uniform(0, 1e-10, size=len(gene_list[duplicate_mask]))
  gene_list.loc[duplicate_mask] = gene_list.loc[duplicate_mask] + noise
  gene_lists[comparison] = gene_list.sort_values(ascending=False)
# gene lists with noise are saved to local memory for GSEA but not written to file

# Note: takes a few minutes to run
for comparison in Comparisons: 
  # run GSEA
  out_path = os.path.join(OUT_GSEA, comparison)  
  gsea_res = gsp.prerank(
    rnk=gene_lists[comparison],
    min_size=15,
    max_size=500,
    gene_sets=GSEA_LIBS,
    permutation_num=PERMUTATIONS,
    outdir=out_path,
    format='png',
    seed=42
  )

In [26]:
# clean up GSEA results
for comparison in Comparisons:
  in_path = os.path.join(OUT_GSEA, comparison, "gseapy.gene_set.prerank.report.csv") 
  df = pd.read_csv(in_path)
  cols_to_keep = ['Term', 'NES', 'FDR q-val', 'Tag %', 'Gene %', 'Lead_genes']
  df = df[cols_to_keep]
  df = df.sort_values(by='NES', ascending=False)
  df[['Database', 'Pathway']] = df['Term'].str.split('__', n=1, expand=True)
  df = df.drop(columns=['Term'])
  df['Direction'] = np.where(df['NES'] > 0, 'Upregulated', 'Downregulated')
  cols = ['Pathway', 'Database', 'NES', 'Direction', 'FDR q-val', 'Tag %', 'Gene %'] 
  df = df[cols]
  out_path = os.path.join(OUT_DIR, f"gsea_{comparison}.csv")
  df.to_csv(out_path, index=False)
